In [1]:
# %%
# 1. 导入所有必要的库
import sys
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Qt5Agg')  # 必须在导入pyplot之前设置
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# PyQt5相关导入
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import *
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qt5agg import NavigationToolbar2QT as NavigationToolbar
# %% [markdown]
# # 🚀 应用程序启动文件

# %%
import sys
from PyQt5.QtWidgets import QApplication
from band_ui import MainWindow  # 注意：现在直接导入 MainWindow
# 确保有这个导入
from band_data import DataManager
from band_algorithms import ClassificationAlgorithms
# sklearn相关导入
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from scipy.ndimage import gaussian_filter1d


import time
from datetime import datetime



In [2]:
class BandSelectionApp(MainWindow):
    """主应用程序"""
    
    def __init__(self):
        super().__init__()
        
        # 初始化管理器
        self.data_manager = DataManager()
        self.classification_algo = ClassificationAlgorithms()
        
        # 当前状态
        self.current_results = None
        self.is_data_loaded = False
        
        # 连接信号槽
        self.connect_signals()
        
        # 初始生成数据
        QTimer.singleShot(500, self.generate_sample_data)
        
    def connect_signals(self):
        """连接信号和槽"""
        # 按钮点击 - 使用新的控件名称
        if hasattr(self, 'generate_data_btn'):
            self.generate_data_btn.clicked.connect(self.generate_sample_data)
        
        if hasattr(self, 'run_analysis_btn'):
            self.run_analysis_btn.clicked.connect(self.run_analysis)
            
        if hasattr(self, 'export_results_btn'):
            self.export_results_btn.clicked.connect(self.export_results)
            
        if hasattr(self, 'reset_all_btn'):
            self.reset_all_btn.clicked.connect(self.clear_data)
            
        # 波段选择变化 - 新的波段选择器
        if hasattr(self, 'band_selector'):
            # 监听波段变化
            for btn in self.band_selector.band_buttons:
                btn.clicked.connect(self.on_bands_changed)
        
    def generate_sample_data(self):
        """生成模拟数据"""
        try:
            # 更新状态 - 使用新的状态栏方法
            self.update_status("正在生成模拟数据...", "processing")
            self.update_progress(20)
            
            # 生成数据
            stats = self.data_manager.generate_sample_data(
                n_samples=800,
                n_features=30,
                n_classes=4
            )
            
            # 显示统计信息
            self.is_data_loaded = True
            
            # 绘制光谱曲线 - 新的绘图方法
            if hasattr(self, 'plot_canvas'):
                self.plot_canvas.plot_spectral(self.data_manager.data)
            
            # 更新数据表格
            self.update_data_table()
            
            # 更新状态
            self.update_progress(100)
            self.update_status("数据生成完成", "success")
            
            # 弹出通知
            QMessageBox.information(self, "成功", 
                f"✅ 模拟数据生成完成！\n"
                f"样本数: {stats['n_samples']}\n"
                f"波段数: {stats['n_features']}\n"
                f"类别数: {stats['n_classes']}")
                
        except Exception as e:
            self.update_status("数据生成失败", "error")
            QMessageBox.critical(self, "错误", f"数据生成失败:\n{str(e)}")
        finally:
            self.update_progress(0)
    
    def update_data_table(self):
        """更新数据预览表格"""
        if self.data_manager.data is not None and hasattr(self, 'data_table'):
            data = self.data_manager.data
            n_bands = data.shape[1]
            
            # 只显示前10个波段
            display_bands = min(10, n_bands)
            self.data_table.setRowCount(display_bands)
            
            for i in range(display_bands):
                band_data = data[:, i]
                
                # 设置表格项
                self.data_table.setItem(i, 0, QTableWidgetItem(f"波段 {i+1}"))
                self.data_table.setItem(i, 1, QTableWidgetItem(f"{band_data.min():.4f}"))
                self.data_table.setItem(i, 2, QTableWidgetItem(f"{band_data.max():.4f}"))
                self.data_table.setItem(i, 3, QTableWidgetItem(f"{band_data.mean():.4f}"))
                self.data_table.setItem(i, 4, QTableWidgetItem(f"{band_data.std():.4f}"))
    
    def run_analysis(self):
        """运行分析"""
        if not self.is_data_loaded:
            QMessageBox.warning(self, "警告", "请先生成数据！")
            return
            
        # 检查波段选择 - 新的波段选择器
        band_selector = getattr(self, 'band_selector', None)
        if not band_selector or not band_selector.selected_bands:
            QMessageBox.warning(self, "警告", "请选择至少一个波段！")
            return
            
        try:
            # 更新状态
            self.update_status("正在进行分析...", "processing")
            
            # 禁用运行按钮
            if hasattr(self, 'run_analysis_btn'):
                self.run_analysis_btn.setEnabled(False)
            
            self.update_progress(30)
            
            # 获取选择的波段数据
            X = self.data_manager.get_subset(band_selector.selected_bands)
            y = self.data_manager.labels
            
            # 获取分类器名称 - 新的方法选择器
            classifier_name = "SVM"  # 默认值
            if hasattr(self, 'method_selector'):
                # 从组合框中获取选择的分类器
                selected_text = self.method_selector.classifier_combo.currentText()
                
                # 映射显示文本到分类器名称
                classifier_mapping = {
                    "支持向量机 (SVM)": "SVM",
                    "随机森林 (Random Forest)": "随机森林",
                    "K近邻 (KNN)": "KNN",
                    "决策树 (Decision Tree)": "决策树",
                    "神经网络 (MLP)": "神经网络",
                    "逻辑回归 (Logistic Regression)": "逻辑回归"
                }
                classifier_name = classifier_mapping.get(selected_text, "SVM")
            
            # 获取测试集比例
            test_size = 0.3
            if hasattr(self, 'method_selector'):
                test_size = self.method_selector.test_size_slider.value() / 100.0
            
            self.update_progress(60)
            
            # 运行分类
            results = self.classification_algo.evaluate_classifier(
                classifier_name, X, y, test_size=test_size
            )
            
            # 保存结果
            self.current_results = results
            
            self.update_progress(90)
            
            # 更新UI显示
            self.update_results_display(results, len(band_selector.selected_bands))
            
            # 更新图表
            self.plot_results()
            
            # 更新状态
            self.update_progress(100)
            self.update_status("分析完成", "success")
            
            QMessageBox.information(self, "完成", 
                f"分析完成！准确率: {results['accuracy']:.4f}")
                
        except Exception as e:
            self.update_status("分析失败", "error")
            QMessageBox.critical(self, "错误", f"分析失败:\n{str(e)}")
            import traceback
            traceback.print_exc()
        finally:
            # 启用运行按钮
            if hasattr(self, 'run_analysis_btn'):
                self.run_analysis_btn.setEnabled(True)
            self.update_progress(0)
    
    def update_results_display(self, results, n_bands):
        """更新结果显示"""
        # 更新结果仪表盘
        if hasattr(self, 'results_dashboard'):
            # 准备结果数据
            display_results = {
                'classifier': results['classifier'],
                'accuracy': results['accuracy'],
                'f1_score': results['f1_score'],
                'precision': results['precision'],
                'recall': results['recall'],
                'run_time': results['run_time'],
                'n_bands': n_bands,
                'n_samples': len(results['y_test']),
                'confusion_matrix_shape': f"{results['confusion_matrix'].shape[0]}x{results['confusion_matrix'].shape[1]}"
            }
            self.results_dashboard.update_results(display_results)
    
    def plot_results(self):
        """绘制结果图表"""
        if not self.current_results or not hasattr(self, 'plot_canvas'):
            return
            
        # 根据当前选择的图表类型绘制
        plot_type = self.plot_canvas.plot_type_combo.currentText()
        
        if plot_type == "光谱曲线":
            if hasattr(self, 'band_selector'):
                self.plot_canvas.plot_spectral(
                    self.data_manager.data,
                    self.band_selector.selected_bands
                )
        elif plot_type == "混淆矩阵":
            self.plot_confusion_matrix()
        elif plot_type == "特征重要性":
            self.plot_feature_importance()
        elif plot_type == "性能对比":
            self.plot_performance_comparison()
    
    def plot_confusion_matrix(self):
        """绘制混淆矩阵"""
        self.plot_canvas.figure.clear()
        ax = self.plot_canvas.figure.add_subplot(111)
        
        cm = self.current_results['confusion_matrix']
        im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
        self.plot_canvas.figure.colorbar(im, ax=ax)
        
        ax.set_title('混淆矩阵', fontweight='bold', pad=12)
        ax.set_xlabel('预测标签', fontsize=11)
        ax.set_ylabel('真实标签', fontsize=11)
        
        # 添加数值标签
        thresh = cm.max() / 2.
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, format(cm[i, j], 'd'),
                       ha="center", va="center",
                       color="white" if cm[i, j] > thresh else "black")
        
        self.plot_canvas.figure.tight_layout()
        self.plot_canvas.canvas.draw()
    
    def plot_feature_importance(self):
        """绘制特征重要性"""
        self.plot_canvas.figure.clear()
        ax = self.plot_canvas.figure.add_subplot(111)
        
        # 获取选择的波段
        selected_bands = getattr(self.band_selector, 'selected_bands', [])
        
        if selected_bands:
            # 模拟特征重要性（实际应从分类器获取）
            importance = np.random.rand(len(selected_bands))
            importance = importance / importance.sum()  # 归一化
            
            x_pos = np.arange(len(selected_bands))
            bars = ax.bar(x_pos, importance, color='#3498DB', alpha=0.8)
            
            ax.set_title('波段重要性', fontweight='bold', pad=12)
            ax.set_xlabel('波段序号', fontsize=11)
            ax.set_ylabel('重要性', fontsize=11)
            ax.set_xticks(x_pos)
            ax.set_xticklabels([f'B{band+1}' for band in selected_bands], rotation=45)
            ax.grid(True, alpha=0.3, axis='y', linestyle='--')
            
            # 添加数值标签
            for bar, value in zip(bars, importance):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{value:.3f}', ha='center', va='bottom', fontsize=9)
        
        self.plot_canvas.figure.tight_layout()
        self.plot_canvas.canvas.draw()
    
    def plot_performance_comparison(self):
        """绘制性能对比图"""
        self.plot_canvas.figure.clear()
        ax = self.plot_canvas.figure.add_subplot(111)
        
        if self.current_results:
            metrics = ['准确率', 'F1得分', '精确率', '召回率']
            values = [
                self.current_results['accuracy'],
                self.current_results['f1_score'],
                self.current_results['precision'],
                self.current_results['recall']
            ]
            
            colors = ['#2ECC71', '#3498DB', '#9B59B6', '#E74C3C']
            x_pos = np.arange(len(metrics))
            
            bars = ax.bar(x_pos, values, color=colors, alpha=0.8, width=0.6)
            
            ax.set_title('性能指标对比', fontweight='bold', pad=12)
            ax.set_ylabel('分数', fontsize=11)
            ax.set_ylim([0, 1])
            ax.set_xticks(x_pos)
            ax.set_xticklabels(metrics, fontsize=11)
            ax.grid(True, alpha=0.3, axis='y', linestyle='--')
            
            # 添加数值标签
            for bar, value in zip(bars, values):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{value:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        self.plot_canvas.figure.tight_layout()
        self.plot_canvas.canvas.draw()
    
    def export_results(self):
        """导出结果"""
        if not self.current_results:
            QMessageBox.warning(self, "警告", "没有可导出的结果！")
            return
            
        # 选择保存路径
        from PyQt5.QtWidgets import QFileDialog
        filepath, _ = QFileDialog.getSaveFileName(
            self, "保存结果", "", "CSV文件 (*.csv);;Excel文件 (*.xlsx);;文本文件 (*.txt)"
        )
        
        if filepath:
            try:
                # 获取选择的波段
                selected_bands = []
                if hasattr(self, 'band_selector'):
                    selected_bands = self.band_selector.selected_bands
                
                # 创建结果DataFrame
                results_df = pd.DataFrame([{
                    '分类器': self.current_results['classifier'],
                    '准确率': self.current_results['accuracy'],
                    'F1分数': self.current_results['f1_score'],
                    '精确率': self.current_results['precision'],
                    '召回率': self.current_results['recall'],
                    '运行时间(s)': self.current_results['run_time'],
                    '选择波段': str(selected_bands),
                    '波段数量': len(selected_bands),
                    '测试集大小': len(self.current_results['y_test']),
                    '时间戳': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
                }])
                
                # 保存文件
                if filepath.endswith('.xlsx'):
                    results_df.to_excel(filepath, index=False)
                elif filepath.endswith('.txt'):
                    results_df.to_csv(filepath, index=False, sep='\t')
                else:
                    results_df.to_csv(filepath, index=False, encoding='utf-8-sig')
                
                QMessageBox.information(self, "成功", f"结果已保存到:\n{filepath}")
                
            except Exception as e:
                QMessageBox.critical(self, "错误", f"导出失败:\n{str(e)}")
    
    def clear_data(self):
        """清空数据"""
        reply = QMessageBox.question(self, '确认', 
                                    '确定要清空所有数据吗？',
                                    QMessageBox.Yes | QMessageBox.No, 
                                    QMessageBox.No)
        
        if reply == QMessageBox.Yes:
            self.data_manager = DataManager()
            self.current_results = None
            self.is_data_loaded = False
            
            # 清空波段选择
            if hasattr(self, 'band_selector'):
                self.band_selector.selected_bands.clear()
                for btn in self.band_selector.band_buttons:
                    btn.setChecked(False)
            
            # 清空数据表格
            if hasattr(self, 'data_table'):
                self.data_table.setRowCount(0)
            
            # 清空图表
            if hasattr(self, 'plot_canvas'):
                self.plot_canvas.figure.clear()
                self.plot_canvas.canvas.draw()
            
            # 重置结果展示
            if hasattr(self, 'results_dashboard'):
                self.results_dashboard.timestamp_label.setText("最后运行: 从未运行")
                self.results_dashboard.detail_text.setText("等待分析结果...")
                
                # 重置指标卡片
                self.results_dashboard.accuracy_card.update_value("0.0000")
                self.results_dashboard.f1_card.update_value("0.0000")
                self.results_dashboard.precision_card.update_value("0.0000")
                self.results_dashboard.recall_card.update_value("0.0000")
            
            self.update_status("系统已重置", "info")
    
    def on_bands_changed(self):
        """波段选择变化时更新图表"""
        if self.is_data_loaded and hasattr(self, 'band_selector') and hasattr(self, 'plot_canvas'):
            selected_bands = self.band_selector.selected_bands
            
            if selected_bands:
                # 更新图表类型为光谱曲线
                if hasattr(self.plot_canvas, 'plot_type_combo'):
                    self.plot_canvas.plot_type_combo.setCurrentText("光谱曲线")
                
                # 绘制光谱曲线
                self.plot_canvas.plot_spectral(
                    self.data_manager.data,
                    selected_bands
                )

In [3]:

# %%
def main():
    """主函数"""
    app = QApplication(sys.argv)
    
    # 设置应用程序样式
    app.setStyle('Fusion')
    
    # 设置应用程序图标
    app.setWindowIcon(QIcon())
    
    # 创建并显示主窗口
    window = BandSelectionApp()
    window.show()
    
    sys.exit(app.exec_())

# %% [markdown]
# ## 6. 🚀 启动应用
# 
# 运行下面的单元格来启动应用程序：

# %%
if __name__ == "__main__":
    main()

🎲 生成模拟数据: 800样本 × 30波段 × 4类别


SystemExit: 0